# Task 2: Data Preprocessing for Regression

## Section 1: Data Cleaning

### 1a. Load Data & Initial Inspection

- Load the dataset and check its shape and column types before any cleaning

In [40]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/final_internship_data.csv")
df.shape

(500000, 26)

In [41]:
df.dtypes

,0
User ID,object
User Name,object
Driver Name,object
Car Condition,object
Weather,object
Traffic Condition,object
key,object
fare_amount,float64
pickup_datetime,object
pickup_longitude,float64


### 1b. Handling Missing Values

- Check missingness percentage per column
- Decides whether imputation is needed at all

In [42]:
df.isnull().mean().sort_values(ascending=False) * 100

,0
dropoff_latitude,0.001
bearing,0.001
jfk_dist,0.001
ewr_dist,0.001
lga_dist,0.001
sol_dist,0.001
distance,0.001
nyc_dist,0.001
dropoff_longitude,0.001
User ID,0.000


- Missingness is negligible (~0.001%) across all columns, with no apparent pattern linking missingness to other features
- Therefore it is reasonable to assume MCAR (Missing Completely At Random)
- Decision: since the amount is negligible and appears random, no imputation was performed

### 1c. Handling Duplicates

- Check exact duplicate rows and duplicate trip IDs (`key`)

In [43]:
print(df.duplicated().sum())
print(df['key'].duplicated().sum())

0
0


- Zero exact duplicates, zero duplicate keys
- Decision: no rows dropped at this step

### 1d. Detecting Invalid / Impossible Values

- Check for physically impossible values: negative/zero fares, zero passengers, unrealistic distances, out-of-range coordinates
- These are data errors, not true missing values or true outliers

In [44]:
print(df['fare_amount'].describe())
print(df['distance'].describe())
print(df['jfk_dist'].describe())
print(df['ewr_dist'].describe())
print(df['lga_dist'].describe())
print(df['sol_dist'].describe())
print(df['nyc_dist'].describe())
print(df['passenger_count'].describe())
print(df['bearing'].describe())
print(df['year'].describe())
print(df[['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude']].describe())

count    500000.000000
mean         11.358361
std           9.916617
min         -44.900000
25%           6.000000
50%           8.500000
75%          12.500000
max         500.000000
Name: fare_amount, dtype: float64
count    499995.000000
mean         19.468775
std         367.299601
min           0.000000
25%           1.214550
50%           2.116970
75%           3.890070
max       12399.956433
Name: distance, dtype: float64
count    499995.000000
mean        385.279367
std        2419.087483
min           1.017646
25%          41.341514
50%          42.523163
75%          43.785649
max       30133.067880
Name: jfk_dist, dtype: float64
count    499995.000000
mean        380.503657
std        2428.804740
min           1.460945
25%          32.173712
50%          34.787507
75%          38.304502
max       30167.595967
Name: ewr_dist, dtype: float64
count    499995.000000
mean        363.843772
std        2425.075903
min           0.382119
25%          17.100762
50%          19.591554

In [45]:
print((df['distance'] < 0).sum())
print((df['fare_amount'] <= 0).sum())
print((df['passenger_count'] == 0).sum())

for col in ['jfk_dist','ewr_dist','lga_dist','sol_dist','nyc_dist']:
    print(col, (df[col] < 0).sum())

for col in ['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude']:
    print(col, ((df[col] < -np.pi) | (df[col] > np.pi)).sum())

0
35
1796
jfk_dist 0
ewr_dist 0
lga_dist 0
sol_dist 0
nyc_dist 0
pickup_longitude 8
pickup_latitude 6
dropoff_longitude 6
dropoff_latitude 5


### 1e. Removing Invalid Rows

- Combine all invalid conditions: distances over 200km (across all five distance columns, not just `jfk_dist`/`distance`), fare <= 0, zero passengers, out-of-range coordinates
- All five distance columns (`jfk_dist`, `ewr_dist`, `lga_dist`, `sol_dist`, `nyc_dist`) can independently contain corrupted values from bad source coordinates, so all five must be checked, not just one
- ~2.45% of rows (12,273) are affected, small enough to drop rather than impute
- Decision: drop, since these represent data errors, not legitimate missing/outlier data

In [46]:
bad_jfk_dist = df['jfk_dist'] > 200
bad_ewr_dist = df['ewr_dist'] > 200
bad_lga_dist = df['lga_dist'] > 200
bad_sol_dist = df['sol_dist'] > 200
bad_nyc_dist = df['nyc_dist'] > 200
bad_distance = df['distance'] > 200
bad_fare = df['fare_amount'] <= 0
bad_passengers = df['passenger_count'] == 0
bad_pickup_long = (df['pickup_longitude'] < -np.pi) | (df['pickup_longitude'] > np.pi)
bad_pickup_lat = (df['pickup_latitude'] < -np.pi) | (df['pickup_latitude'] > np.pi)

all_invalid = (
    bad_jfk_dist | bad_ewr_dist | bad_lga_dist | bad_sol_dist | bad_nyc_dist |
    bad_distance | bad_fare | bad_passengers | bad_pickup_long | bad_pickup_lat
)
print("total invalid rows:", all_invalid.sum())

total invalid rows: 12273


In [47]:
df = df[~all_invalid].reset_index(drop=True)
df.shape

(487727, 26)

### 1f-checkpoint. Snapshot Before Any Reshaping

- `df_raw` is a checkpoint of the data right after invalid-row removal (1e), before radians→degrees, column drops, or feature engineering
- Everything from here through Section 2 (1f, 2a-2h) is exploratory: it's how the drop/keep/encode decisions below were justified, and it keeps mutating `df` for that narrative
- The actual reproducible transforms live in Section 4 as sklearn transformers, built to run on `df_raw` (via `X_train`/`X_test`) so the full pipeline reproduces every decision below automatically — no manual steps required outside the pipeline

In [48]:
df_raw = df.copy()
df_raw.shape

(487727, 26)

### 1f. Fixing Unit Errors (Radians -> Degrees)

- Pickup/dropoff latitude and longitude are stored in radians, convert to degrees
- Bearing is stored as a signed radian angle (-pi to pi), convert to degrees in the standard 0-360 degree range

In [49]:
coord_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
for col in coord_cols:
    df[col] = np.degrees(df[col])

df['bearing'] = np.degrees(df['bearing'])
df['bearing'] = (df['bearing'] + 360) % 360

df[coord_cols + ['bearing']].describe()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,bearing
count,487727.000000,487727.000000,487727.000000,487727.000000,487727.000000
mean,-73.975383,40.750817,-73.974397,40.751211,203.853079
std,0.037653,0.029520,0.036838,0.032593,105.614596
min,-75.437825,39.603178,-75.448579,39.603429,0.000000
25%,-73.992267,40.736543,-73.991569,40.735600,134.146611
50%,-73.982080,40.753376,-73.980583,40.753874,186.320674
75%,-73.968343,40.767469,-73.965284,40.768407,314.983294
max,-73.085745,41.650000,-72.196091,41.505343,359.998528


### 1g. Outlier Detection (IQR) on `fare_amount` and `distance`

- Now that impossible values are removed, check for statistical outliers using the IQR rule
- This is separate from 1e: those were data errors, these are extreme-but-possibly-real values

In [50]:
for col in ['fare_amount', 'distance']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(col, "lower:", lower, "upper:", upper, "outliers:", n_outliers)

fare_amount lower: -3.75 upper: 22.25 outliers: 42212
distance lower: -2.752733813031015 upper: 7.938053224076305 outliers: 40549


- Both `fare_amount` and `distance` are heavily right-skewed (fare skew = 4.87), so the IQR rule flags a large share of legitimate long/expensive trips as "outliers"
- These are not data errors (already removed in 1e) , they are the real, rare-but-valid data

### 1h. Outlier Handling Decision

- Decision: **keep all rows**, no deletion , these are real trips, not data errors, and removing this much data would bias the model against learning the expensive/long-distance cases
- Instead of deleting, outliers will be **capped (winsorized)**, not removed
- Capping will use custom percentile clipping (1st/99th percentile) for explicit control per column
- This must be fit on training data only and applied to test data with the same thresholds, so it belongs in Section 4, not here , computing percentiles on the full dataset before splitting would leak test information into training

## Section 2: Feature Engineering & Selection

### 2a. Dropping Redundant Datetime Column

- `pickup_datetime` is redundant since hour, day, month, weekday, and year are already extracted as separate columns

In [51]:
df = df.drop(columns=['pickup_datetime'])
df.shape

(487727, 25)

### 2b. Verifying `day` vs `weekday` Are Distinct

- Check whether these two columns are duplicates of each other before deciding to drop either

In [52]:
print(df['day'].unique())
print(df['weekday'].unique())

[15  5 18 21  9  6 20  4  3  2  8 19 22  7 12 10 28 11 24 29 31  1 14 23
 16 17 27 25 30 26 13]
[0 1 3 5 2 6 4]


- `day` = day of month (1-31), `weekday` = day of week (0-6)
- Genuinely different fields, not duplicates

### 2c. Target Skewness Check (`fare_amount`)

- Check the shape of the target distribution to decide if a transform is needed

In [53]:
print(df['fare_amount'].skew())
df['fare_amount'].describe()

4.8690474106057975


,fare_amount
count,487727.000000
mean,11.352540
std,9.855237
min,0.010000
25%,6.000000
50%,8.500000
75%,12.500000
max,500.000000


- Heavily right-skewed
- Decision: apply `log1p` to the target at modeling time (Section 4), not here

### 2d. Correlation Among Distance Features

- Check for redundancy between the five distance columns before feature selection

In [54]:
dist_cols = ['jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist', 'distance']
df[dist_cols].corr()

,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance
jfk_dist,1.000000,-0.247834,0.102122,-0.043307,-0.071941,-0.247710
ewr_dist,-0.247834,1.000000,-0.246280,0.958898,0.944480,0.496832
lga_dist,0.102122,-0.246280,1.000000,-0.255991,-0.154768,0.260625
sol_dist,-0.043307,0.958898,-0.255991,1.000000,0.985688,0.438717
nyc_dist,-0.071941,0.944480,-0.154768,0.985688,1.000000,0.491259
distance,-0.247710,0.496832,0.260625,0.438717,0.491259,1.000000


- `ewr_dist`, `sol_dist`, `nyc_dist` are highly correlated (0.95-0.99), near redundant
- Decision: keep `jfk_dist` and `lga_dist` (distinct info), keep `nyc_dist` only (most general), drop `ewr_dist` and `sol_dist`

In [55]:
df = df.drop(columns=['ewr_dist', 'sol_dist'])
df.shape

(487727, 23)

### 2e. Correlation With Target (Feature Relevance)

- Check correlation of all numeric columns with `fare_amount` to spot weak predictors

In [56]:
df.corr(numeric_only=True)['fare_amount'].sort_values(ascending=False)

,fare_amount
fare_amount,1.000000
distance,0.756355
nyc_dist,0.396454
pickup_longitude,0.379927
dropoff_longitude,0.304516
lga_dist,0.123101
year,0.115943
month,0.024467
passenger_count,0.013329
weekday,0.003196


### 2f. Categorical Features vs Fare

- Check average fare across categories for `Car Condition`, `Weather`, `Traffic Condition`

In [57]:
print(df.groupby('Car Condition')['fare_amount'].mean())
print(df.groupby('Weather')['fare_amount'].mean())
print(df.groupby('Traffic Condition')['fare_amount'].mean())

Car Condition
Bad          11.318373
Excellent    11.345831
Good         11.332621
Very Good    11.413207
Name: fare_amount, dtype: float64
Weather
cloudy    11.373461
rainy     11.345270
stormy    11.339628
sunny     11.358133
windy     11.346161
Name: fare_amount, dtype: float64
Traffic Condition
Congested Traffic    11.386091
Dense Traffic        11.358103
Flow Traffic         11.313356
Name: fare_amount, dtype: float64


- Group-wise mean fares showed minimal variation across categories for all three variables, indicating a weak predictive relationship with the target
- Retaining them would increase dimensionality without meaningful predictive gain
- Decision: drop all three rather than encoding them

In [58]:
df = df.drop(columns=['Car Condition', 'Weather', 'Traffic Condition'])
df.shape

(487727, 20)

### 2g. Dropping Identifiers & Low-Signal Features

- Identifier columns (`User ID`, `User Name`, `Driver Name`, `key`) have no predictive value
- `passenger_count` has negligible correlation with fare (0.013)

In [59]:
df = df.drop(columns=['User ID', 'User Name', 'Driver Name', 'key', 'passenger_count'])
df.shape

(487727, 15)

### 2h. Cyclical Encoding

- `hour`, `weekday`, `month`, and `bearing` are circular variables, e.g. hour 23 and hour 0 are close in reality but raw integers treat them as far apart
- Apply sin/cos encoding, then drop the raw columns

In [60]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['bearing_sin'] = np.sin(2 * np.pi * df['bearing'] / 360)
df['bearing_cos'] = np.cos(2 * np.pi * df['bearing'] / 360)

df = df.drop(columns=['hour', 'weekday', 'month', 'bearing'])
df.shape

(487727, 19)

## Section 3: Train/Test Split

### 3a. Train/Test Split

- Split into train/test before any capping, fitting, or transforming — every later step must only ever learn from `X_train`
- Simple i.i.d. split is appropriate here (not time-series data)
- `test_size=0.2`, `random_state=42` for reproducibility

In [61]:
from sklearn.model_selection import train_test_split

# Split on df_raw (post-cleaning, pre-transform) so the pipeline in Section 4
# reproduces every transform/drop decision automatically from raw input
X = df_raw.drop(columns=['fare_amount'])
y = df_raw['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(390181, 25) (97546, 25)


## Section 4: Preprocessing Pipeline

### 4a. Custom Transformers for Full Reproducible Preprocessing

- Everything decided in 1f and Section 2 (radians→degrees, dropped columns, cyclical encoding, outlier capping) is now implemented.

### 4b. Degree Converter (Radians → Degrees, from 1f)

- Converts pickup/dropoff lat-long from radians to degrees
- Converts `bearing` from a signed radian angle to the standard 0-360 degree range
- Stateless (no `fit` logic needed), but implemented as a transformer so it slots into the `Pipeline`

In [62]:
from sklearn.base import BaseEstimator, TransformerMixin

class DegreeConverter(BaseEstimator, TransformerMixin):
    def __init__(self, coord_cols, bearing_col='bearing'):
        self.coord_cols = coord_cols
        self.bearing_col = bearing_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.coord_cols:
            X[col] = np.degrees(X[col])
        if self.bearing_col in X.columns:
            X[self.bearing_col] = np.degrees(X[self.bearing_col])
            X[self.bearing_col] = (X[self.bearing_col] + 360) % 360
        return X

### 4c. Column Dropper (Redundant / Low-correlation / Identifier Columns)

Drops, in one place, every column that Section 2 decided to remove:
- `pickup_datetime` — redundant once hour/day/month/weekday/year are extracted (2a)
- `ewr_dist`, `sol_dist` — near-redundant with `nyc_dist`, corr 0.95-0.99 (2d)
- `Car Condition`, `Weather`, `Traffic Condition` — negligible relationship with fare (2f)
- `User ID`, `User Name`, `Driver Name`, `key` — identifiers, no predictive value (2g)
- `passenger_count` — negligible correlation with fare, 0.013 (2g)

In [63]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.columns if c in X.columns])

### 4d. Cyclical Encoder (hour / weekday / month / bearing, from 2h)

- Sin/cos-encodes the circular variables so e.g. hour 23 and hour 0 end up close together
- Drops the raw columns after encoding, same as the manual version in 2h
- Must run after the `DegreeConverter`, since `bearing`'s sin/cos needs it in the 0-360 degree range

In [64]:
class CyclicalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, period_map):
        # e.g. {'hour': 24, 'weekday': 7, 'month': 12, 'bearing': 360}
        self.period_map = period_map

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col, period in self.period_map.items():
            X[f'{col}_sin'] = np.sin(2 * np.pi * X[col] / period)
            X[f'{col}_cos'] = np.cos(2 * np.pi * X[col] / period)
        X = X.drop(columns=list(self.period_map.keys()))
        return X

### 4e. Percentile Capper (Outlier Handling, from 1h)

- Implements the capping decision from Section 1h: clip outliers in `distance` at the 1st/99th percentile
- `fit()` only ever sees `X_train`.

In [65]:
class PercentileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, columns, lower_pct=0.01, upper_pct=0.99):
        self.columns = columns
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        self.bounds_ = {}
        for col in self.columns:
            lower = X[col].quantile(self.lower_pct)
            upper = X[col].quantile(self.upper_pct)
            self.bounds_[col] = (lower, upper)
        return self

    def transform(self, X):
        X = X.copy()
        for col, (lower, upper) in self.bounds_.items():
            X[col] = X[col].clip(lower, upper)
        return X

### 4f. Assemble & Fit the Full Preprocessing Pipeline

In [66]:
from sklearn.pipeline import Pipeline

coord_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']

drop_cols = [
    'pickup_datetime',                                  # 2a
    'ewr_dist', 'sol_dist',                              # 2d
    'Car Condition', 'Weather', 'Traffic Condition',     # 2f
    'User ID', 'User Name', 'Driver Name', 'key',        # 2g
    'passenger_count'                                    # 2g
]

cyclical_periods = {'hour': 24, 'weekday': 7, 'month': 12, 'bearing': 360}

preprocessing_pipeline = Pipeline([
    ('degree_converter', DegreeConverter(coord_cols=coord_cols)),
    ('column_dropper', ColumnDropper(columns=drop_cols)),
    ('cyclical_encoder', CyclicalEncoder(period_map=cyclical_periods)),
    ('capper', PercentileCapper(columns=['distance']))
])

preprocessing_pipeline.fit(X_train)

X_train_final = preprocessing_pipeline.transform(X_train)
X_test_final = preprocessing_pipeline.transform(X_test)

X_train_final.shape, X_test_final.shape

((390181, 18), (97546, 18))

### 4g. Log1p Transform on Target (`fare_amount`)

- Applying `log1p` to compress the right-skew found in Section 2c

- Predictions will need `np.expm1()` applied back before computing MAE/RMSE in real dollar units (done at evaluation time)

In [67]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(y_train.skew())
print(y_train_log.skew())

4.9976635317120515
0.9840494936073562


### 4h. Note on Encoding & Scaling

- **Encoding:** all categorical columns (`Car Condition`, `Weather`, `Traffic Condition`) are dropped by `ColumnDropper` for showing no meaningful relationship with `fare_amount` (2f). The remaining predictors are entirely numeric after the pipeline runs — therefore no encoding step is needed
- **Scaling:** Decision Trees and XGBoost are tree-based models whose splits are invariant to scaling .Therefore `StandardScaler`/`MinMaxScaler` were intentionally not done, not forgotten

## Section 5: Decision Tree — Baseline & Tuning

### 5a. Baseline Model: Decision Tree Regressor (Untuned)

- Trained with default hyperparameters , no tuning yet. This is the true baseline: the simplest honest starting point before any optimization


In [68]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

dt_baseline = DecisionTreeRegressor(random_state=42)

dt_baseline_scores = cross_val_score(
    dt_baseline,
    X_train_final,
    y_train_log,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print("Per-fold RMSE (log scale):", -dt_baseline_scores)
print("Mean RMSE (log scale):", -dt_baseline_scores.mean())

Per-fold RMSE (log scale): [0.31303244 0.31453757 0.31139091 0.30970599 0.30962963]
Mean RMSE (log scale): 0.31165931012619674


### 5b. Fit Baseline on Full Training Set


In [69]:
dt_baseline.fit(X_train_final, y_train_log)

DecisionTreeRegressor(random_state=42)

### 5c. Hyperparameter Tuning: Decision Tree

- Search over the parameters that control a tree's complexity and overfitting behavior:
  - `max_depth` — how deep the tree can grow
  - `min_samples_split` — minimum samples required to split a node (higher = less overfitting)
  - `min_samples_leaf` — minimum samples required in a leaf (higher = smoother, less noisy splits)
  - `max_features` — how many features are considered at each split (adds randomness, reduces variance)
- Using `RandomizedSearchCV`

In [70]:
from sklearn.model_selection import RandomizedSearchCV

dt_param_grid = {
    'max_depth': [4, 6, 8, 10, 12, 15],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features': [None, 'sqrt', 'log2']
}

dt_search = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=dt_param_grid,
    n_iter=20,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

dt_search.fit(X_train_final, y_train_log)

print(dt_search.best_params_)
print(dt_search.best_score_)

{'min_samples_split': 5, 'min_samples_leaf': 20, 'max_features': None, 'max_depth': 10}
-0.23505764497722553


## Section 6: XGBoost — Tuned Model

### 6a. Hyperparameter Tuning: XGBoost Regressor

- Tune the parameters that most affect XGBoost's accuracy/overfitting tradeoff:
  - `n_estimators` — number of boosting rounds
  - `max_depth` — depth of each individual tree (kept shallower than a standalone tree, since boosting adds many of them)
  - `learning_rate` — how much each tree corrects the previous ones
  - `subsample` — fraction of rows used per tree (adds regularization)
  - `colsample_bytree` — fraction of features used per tree (adds regularization)
- Same `RandomizedSearchCV` approach as the Decision Tree

In [72]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Hyperparameter search space
xgb_param_grid = {
    'n_estimators': [200, 300, 500, 700, 1000],
    'max_depth': [3, 4, 5, 6, 7, 8, 10],
    'learning_rate': [0.001, 0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# Hyperparameter tuning
xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        tree_method="hist",
        device="cuda",
        n_jobs=-1
    ),
    param_distributions=xgb_param_grid,
    n_iter=30,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1
)

xgb_search.fit(X_train_final, y_train_log)

print("Best Parameters:", xgb_search.best_params_)
print("Best CV Score:", xgb_search.best_score_)

# Retrain best model using Early Stopping

X_train_es, X_val, y_train_es, y_val = train_test_split(
    X_train_final,
    y_train_log,
    test_size=0.2,
    random_state=42
)

xgb_best = XGBRegressor(
    **xgb_search.best_params_,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=10
)

xgb_best.fit(
    X_train_es,
    y_train_es,
    eval_set=[(X_val, y_val)],
    verbose=False
)



/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [18:43:02] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Best Parameters: {'subsample': 0.8, 'n_estimators': 700, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
Best CV Score: -0.20772033890750033


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=10,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=700,
             n_jobs=-1, num_parallel_tree=None, ...)

In [73]:
# Test set evaluation
xgb_preds_log = xgb_best.predict(X_test_final)
xgb_preds = np.expm1(xgb_preds_log)
y_test_actual = np.expm1(y_test_log)

print("\nTest Metrics After Early Stopping")
print("MAE :", mean_absolute_error(y_test_actual, xgb_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual, xgb_preds)))
print("R²  :", r2_score(y_test_actual, xgb_preds))

print("\nBest Iteration:", xgb_best.best_iteration)
print("Best Validation Score:", xgb_best.best_score)


Test Metrics After Early Stopping
MAE : 1.5429681910258743
RMSE: 4.0786419092223385
R²  : 0.8256198963065005

Best Iteration: 512
Best Validation Score: 0.20606309348550048


## Section 7: Model Comparison

### 7a. Comparison: Decision Tree (Baseline vs Tuned) and XGBoost (Tuned)

- Evaluate all three models on the **untouched test set** , first time any of them touch test data

- MAE, RMSE, R² computed per the task's required metrics

In [74]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

dt_tuned_best = dt_search.best_estimator_
xgb_best = xgb_search.best_estimator_

dt_baseline_preds_log = dt_baseline.predict(X_test_final)
dt_tuned_preds_log = dt_tuned_best.predict(X_test_final)
xgb_preds_log = xgb_best.predict(X_test_final)

dt_baseline_preds = np.expm1(dt_baseline_preds_log)
dt_tuned_preds = np.expm1(dt_tuned_preds_log)
xgb_preds = np.expm1(xgb_preds_log)

y_test_actual = np.expm1(y_test_log)

results = pd.DataFrame({
    'Model': ['Decision Tree (untuned baseline)', 'Decision Tree (tuned)', 'XGBoost (tuned)'],
    'MAE': [
        mean_absolute_error(y_test_actual, dt_baseline_preds),
        mean_absolute_error(y_test_actual, dt_tuned_preds),
        mean_absolute_error(y_test_actual, xgb_preds)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test_actual, dt_baseline_preds)),
        np.sqrt(mean_squared_error(y_test_actual, dt_tuned_preds)),
        np.sqrt(mean_squared_error(y_test_actual, xgb_preds))
    ],
    'R2': [
        r2_score(y_test_actual, dt_baseline_preds),
        r2_score(y_test_actual, dt_tuned_preds),
        r2_score(y_test_actual, xgb_preds)
    ]
})

results

,Model,MAE,RMSE,R2
0,Decision Tree (untuned baseline),2.437986,5.643745,0.666112
1,Decision Tree (tuned),1.872224,4.308226,0.805436
2,XGBoost (tuned),1.526623,4.060507,0.827167


## Section 8: Final Combined Pipeline

### 8a. Final Combined Pipeline (Preprocessing + Model)

- One single fitted `Pipeline` object combining all four preprocessing units (Section 4) and the final model
- Fit directly on `X_train`/`y_train` as they come straight out of the split

- `TransformedTargetRegressor` folds in the `log1p`/`expm1` step: it automatically applies `log1p` to `y` before fitting, and automatically applies `expm1` to predictions on `.predict()`
- Uses `xgb_search.best_params_` (the winning hyperparameters from Section 6), so this final pipeline reflects the tuned model
- This is the single object saved with `joblib`

In [75]:
from sklearn.compose import TransformedTargetRegressor
import joblib

final_model = final_model = XGBRegressor(
    **xgb_search.best_params_,
    random_state=42,
    n_jobs=-1
)

target_transformed_model = TransformedTargetRegressor(
    regressor=final_model,
    func=np.log1p,
    inverse_func=np.expm1
)

final_pipeline = Pipeline([
    ('degree_converter', DegreeConverter(coord_cols=coord_cols)),
    ('column_dropper', ColumnDropper(columns=drop_cols)),
    ('cyclical_encoder', CyclicalEncoder(period_map=cyclical_periods)),
    ('capper', PercentileCapper(columns=['distance'])),
    ('model', target_transformed_model)
])

final_pipeline.fit(X_train, y_train)

joblib.dump(final_pipeline, 'fare_prediction_pipeline.pkl')

print("Pipeline saved.")

Pipeline saved.


### 8b. Sanity Check: Load & Predict with the Saved Pipeline

In [76]:
loaded_pipeline = joblib.load('fare_prediction_pipeline.pkl')

pipeline_preds = loaded_pipeline.predict(X_test)

pipeline_mae = mean_absolute_error(y_test, pipeline_preds)
pipeline_rmse = np.sqrt(mean_squared_error(y_test, pipeline_preds))
pipeline_r2 = r2_score(y_test, pipeline_preds)

print("MAE:", pipeline_mae)
print("RMSE:", pipeline_rmse)
print("R2:", pipeline_r2)

MAE: 1.527703910864827
RMSE: 4.057070040790016
R2: 0.8274596051808178


## Section 9: Final Notes

### 9a. What I Would Try Next

- **Try target/frequency encoding on a re-included categorical feature** , `Car Condition`, `Weather`, and `Traffic Condition` were dropped for showing no *average* fare difference (2f), but a groupby on the mean can hide interaction effects (e.g. bad weather might only matter combined with rush hour). Worth revisiting with an interaction feature instead of dropping outright

- **Try Gradient Boosting or LightGBM as a third comparison point** , since XGBoost already outperformed the Decision Tree, adding one more boosting-family model would clarify whether XGBoost's specific implementation mattered, or whether any modern boosting method would land in the same range
- **Revisit the capping thresholds** , 1st/99th percentile was a reasonable default, but testing 2nd/98th or 5th/95th on `distance` could reveal whether the cap is too aggressive or too loose for this dataset's tail shape